# Backtest EMA Cross Long Only

Run the EMA Cross Long Only strategy on a simulated exchange venue.

**Strategy logic:**
- **Entry:** BUY when flat and fast EMA ≥ slow EMA (regime-based, not crossover)
- **Exit:** Close long when fast EMA < slow EMA
- Never opens short positions — useful as a baseline or for long-only markets
- After exit, re-enters immediately on the next bar if regime persists

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + EMA overlay with trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweep — EMA fast vs slow period grid
6. Comparison — long-only vs bidirectional EMACross with best params

**Prerequisites:** Run `scripts/fetch_binance_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_single_backtest, run_sweep
from src.core import bar_type_str, with_venue_config
from src.strategies.ema_cross_long_only import EMACrossLongOnly, EMACrossLongOnlyConfig
from src.strategies.ma_cross import MACross, MACrossConfig

from charts import (
    plot_ema_cross, plot_equity_curve, print_summary_stats,
    plot_pnl_heatmap, generate_backtest_html,
)
from utils import make_instrument_id, save_notebook, save_notebook_html, save_tearsheet

# ── Shared config ────────────────────────────────────────────────────
EXCHANGE         = "BINANCE"
ASSET            = "BTC"
BAR_INTERVAL     = "1d"
STARTING_CAPITAL = 10_000
TRADE_SIZE       = 1000              # $1,000 USD per trade
LEVERAGE         = 20                # Backtesting leverage (margin_init = 1/20 = 5%)
SAVE_TEARSHEET   = True

# EMA params
FAST_EMA = 10
SLOW_EMA = 20

# Sweep grids
FAST_PERIODS = [5, 10, 15, 20, 25]
SLOW_PERIODS = [15, 20, 30, 40, 50]

# Standard
TRADE_NOTIONAL = Decimal(TRADE_SIZE)
INSTRUMENT_ID  = make_instrument_id(ASSET, EXCHANGE)
TRADING_PAIR   = INSTRUMENT_ID.split("-")[0]
BAR_TYPE_STR   = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE          = Venue(EXCHANGE)

# Files
CATALOG_PATH = "../data/catalog"
RESULT_NAME  = f"EMACrossLongOnly_{INSTRUMENT_ID}_{BAR_INTERVAL}_f{FAST_EMA}_s{SLOW_EMA}"

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

# Override margins for backtesting (catalog instruments have raw/live margins)
instrument = with_venue_config(instrument, LEVERAGE)

# Settlement currency for PnL queries (USDC for HL, USDT for Binance)
CURRENCY = instrument.settlement_currency

print(f"Instrument : {instrument.id}")
print(f"Currency   : {CURRENCY}")
print(f"Leverage   : {LEVERAGE}x (margin_init={instrument.margin_init}, margin_maint={instrument.margin_maint})")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────────────
engine = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
print("Engine configured.")

In [ ]:
# ── Cell 4: Add EMACrossLongOnly strategy + run ──────────────────────
config = EMACrossLongOnlyConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_notional=TRADE_NOTIONAL,
    fast_ema_period=FAST_EMA,
    slow_ema_period=SLOW_EMA,
)
strategy = EMACrossLongOnly(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Calculate statistics ──────────────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet ────────────────────────────────────────────────
html = create_tearsheet(
    engine,
    output_path=None,
    title=f"EMACrossLongOnly({FAST_EMA}/{SLOW_EMA}) | {INSTRUMENT_ID} {BAR_INTERVAL}",
)
display(HTML(html))

if SAVE_TEARSHEET:
    save_tearsheet(html, RESULT_NAME)

In [ ]:
# ── Cell 8: Price chart with EMA overlay + trade markers ──────────────

fig = plot_ema_cross(
    bars, fills_report,
    fast_period=FAST_EMA,
    slow_period=SLOW_EMA,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity curve ──────────────────────────────────────────────────
plot_equity_curve(
    analyzer, account_report,
    f"EMACrossLongOnly({FAST_EMA}/{SLOW_EMA})  {INSTRUMENT_ID} {BAR_INTERVAL}",
)

In [ ]:
# ── Cell 10: Summary statistics ───────────────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions), currency=CURRENCY)

In [ ]:
# ── Cell 11: Parameter sweep — EMA fast vs slow ───────────────────────

def ema_long_only_factory(eng, params):
    cfg = EMACrossLongOnlyConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_notional=TRADE_NOTIONAL,
        fast_ema_period=params["fast"],
        slow_ema_period=params["slow"],
    )
    eng.add_strategy(EMACrossLongOnly(cfg))

combos = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]

results_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=ema_long_only_factory,
    strategy_name="EMACrossLongOnly",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
)

In [ ]:
# ── Cell 12: PnL heatmap (EMA fast vs slow) ───────────────────────────
plot_pnl_heatmap(
    results_df, row_col="slow", col_col="fast",
    row_label="Slow EMA Period", col_label="Fast EMA Period",
    title=f"Total PnL ({CURRENCY}) by EMA Parameters — Long Only",
)

In [ ]:
# ── Cell 13: Compare long-only vs bidirectional MACross(EMA) ─────────
#
# Run the bidirectional MACross with the best long-only params
# to see if shorting adds or destroys value.

best_row = results_df.loc[results_df["total_pnl"].idxmax()]
best_fast = int(best_row["fast"])
best_slow = int(best_row["slow"])
lo_pnl = best_row["total_pnl"]
lo_pnl_pct = best_row["total_pnl_pct"]
lo_positions = int(best_row["num_positions"])

print(f"Best long-only params: fast={best_fast}, slow={best_slow}")
print(f"  PnL       : {lo_pnl:,.2f} ({lo_pnl_pct:.2f}%)")
print(f"  Positions : {lo_positions}")

# Run bidirectional MACross(EMA) with same params
def bidir_factory(eng, params):
    cfg = MACrossConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_notional=TRADE_NOTIONAL,
        ma_type="EMA",
        fast_period=params["fast"],
        slow_period=params["slow"],
    )
    eng.add_strategy(MACross(cfg))

bidir_result = run_single_backtest(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    params={"fast": best_fast, "slow": best_slow},
    add_strategy=lambda eng: bidir_factory(eng, {"fast": best_fast, "slow": best_slow}),
)

bd_pnl = bidir_result.get("total_pnl", 0)
bd_pnl_pct = bidir_result.get("total_pnl_pct", 0)
bd_positions = bidir_result.get("num_positions", 0)

print(f"\nBidirectional EMACross({best_fast}/{best_slow}):")
print(f"  PnL       : {bd_pnl:,.2f} ({bd_pnl_pct:.2f}%)")
print(f"  Positions : {bd_positions}")

diff = lo_pnl - bd_pnl
print(f"\nLong-only advantage: {diff:+,.2f} ({'+' if diff >= 0 else ''}{lo_pnl_pct - bd_pnl_pct:.2f}%)")
if diff > 0:
    print("  → Short side is hurting. Long-only is better for this instrument/period.")
else:
    print("  → Short side adds value. Bidirectional outperforms.")

In [ ]:
# ── Cell 14: TradingView Interactive Backtest ──────────────────────────

path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=FAST_EMA,
    slow_period=SLOW_EMA,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    result_filename=RESULT_NAME,
)
# Then just open the file in your browser

In [ ]:
# ── Cell 15: Save notebook snapshot ──────────────────────────────────────
#
# Save the notebook (Ctrl+S) before running this cell.

#save_notebook("backtest_ema_cross_long_only.ipynb", RESULT_NAME)
#save_notebook_html("backtest_ema_cross_long_only.ipynb", RESULT_NAME)

In [ ]:
# ── Cell 16: Cleanup ──────────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")